# Time-Dependent Schrödinger Equation — PINN Solution

This notebook applies a Physics-Informed Neural Network to the **1D time-dependent Schrödinger equation**:

$$
i\hbar\frac{\partial \psi}{\partial t} = -\frac{\hbar^2}{2m}\frac{\partial^2 \psi}{\partial x^2} + V(x)\psi
$$

We adopt units $\hbar = m = 1$ and use a harmonic potential $V(x) = \frac{1}{2}x^2$.

The complex wavefunction is split into real and imaginary parts:

$$
\psi(x,t) = \psi_r(x,t) + i\,\psi_i(x,t)
$$

yielding two coupled real PDEs that form the PINN residual. The initial condition is a Gaussian wavepacket:

$$
\psi(x, 0) = \exp\!\left(-\frac{x^2}{2}\right)
$$

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.animation as animation

from src.pinn import ComplexPINN
from src.physics import harmonic_potential, schrodinger_residual_1d, PINNLoss
from src.data import sample_interior, sample_boundary, sample_initial

plt.style.use('dark_background')
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

In [ ]:
# ── Configuration ──────────────────────────────────────────────────────────
X_MIN, X_MAX = -8.0, 8.0
T_MIN, T_MAX = 0.0,  2.0
N_COL    = 4000
N_BC     = 400
N_IC     = 400
N_EPOCHS = 10000
LR       = 1e-3

model = ComplexPINN(in_dim=2, hidden_dim=80, n_layers=5, activation='tanh').to(device)
loss_fn = PINNLoss(lambda_pde=1.0, lambda_bc=20.0, lambda_ic=20.0)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=800, factor=0.5)

print(f'Model parameters: {model.count_parameters():,}')

In [ ]:
# ── Training ───────────────────────────────────────────────────────────────
history = []

for epoch in range(1, N_EPOCHS + 1):
    optimizer.zero_grad()

    x_col, t_col = sample_interior(N_COL, (X_MIN, X_MAX), (T_MIN, T_MAX), device)
    x_bc,  t_bc  = sample_boundary(N_BC,  (X_MIN, X_MAX), (T_MIN, T_MAX), device)
    x_ic,  t_ic  = sample_initial(N_IC,   (X_MIN, X_MAX), 0.0,            device)

    psi_r_col, psi_i_col = model(x_col, t_col)
    res_r, res_i = schrodinger_residual_1d(
        psi_r_col, psi_i_col, x_col, t_col, harmonic_potential)

    psi_r_bc, psi_i_bc = model(x_bc, t_bc)

    psi_r_ic, psi_i_ic = model(x_ic, t_ic)
    ic_r_target = torch.exp(-x_ic**2 / 2)
    ic_i_target = torch.zeros_like(ic_r_target)

    losses = loss_fn(
        pde_residuals=[res_r, res_i],
        bc_residuals=[psi_r_bc, psi_i_bc],
        ic_residuals=[psi_r_ic - ic_r_target, psi_i_ic - ic_i_target],
        data_residuals=[],
    )
    losses['total'].backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()
    scheduler.step(losses['total'].item())

    if epoch % 2000 == 0:
        print(f'Epoch {epoch:6d} | total={losses["total"].item():.3e} '
              f'| pde={losses["pde"].item():.3e} '
              f'| ic={losses["ic"].item():.3e}')
        history.append({'epoch': epoch, 'total': losses['total'].item()})

In [ ]:
# ── Visualization: probability density at several time snapshots ───────────
model.eval()
times = [0.0, 0.5, 1.0, 1.5, 2.0]
x_vis = np.linspace(X_MIN, X_MAX, 400)
x_t = torch.FloatTensor(x_vis).unsqueeze(1).to(device)

fig, axes = plt.subplots(1, len(times), figsize=(16, 3.5), facecolor='#0d1117')
cmap = plt.cm.plasma

for ax, ti in zip(axes, times):
    t_t = torch.full_like(x_t, ti)
    with torch.no_grad():
        density = model.probability_density(x_t, t_t).cpu().numpy().flatten()
    ax.fill_between(x_vis, density, alpha=0.7, color='#58a6ff')
    ax.plot(x_vis, density, color='#79c0ff', linewidth=1.5)
    ax.set_title(f't = {ti:.1f}', color='#e6edf3', fontsize=10)
    ax.set_facecolor('#161b22')
    ax.tick_params(colors='#8b949e')
    ax.set_xlabel('x', color='#8b949e')
    ax.set_ylabel('|ψ|²', color='#8b949e')
    for sp in ax.spines.values(): sp.set_edgecolor('#30363d')

fig.suptitle('Probability Density |ψ(x,t)|² — PINN Solution', color='#e6edf3', y=1.02)
plt.tight_layout()
plt.savefig('../outputs/schrodinger_density_snapshots.png', dpi=150, bbox_inches='tight',
            facecolor='#0d1117')
plt.show()
print('Saved → outputs/schrodinger_density_snapshots.png')